# Pixio Fine-tuning on Custom Data

> Fine-tune Meta's **Pixio** (*Pixel Supervision for Visual Pre-training*) Vision Transformer on your own image dataset.

Pixio (arXiv:2512.15715) is Meta's 2025 dense-prediction-focused ViT, trained via enhanced Masked Autoencoding on 2B web images.  
Its four key improvements over standard MAE:

1. **Deeper decoder** — heavier decoder shifts reconstruction burden off the encoder  
2. **4×4 block masking** — prevents shortcut reconstruction  
3. **8 [CLS] tokens** — richer image-level representations than the usual single token  
4. **2 B web-scale training data** — greater visual diversity

This notebook demonstrates three fine-tuning strategies for **image classification**:

| Strategy | Description |
|---|---|
| `linear_probe` | Freeze backbone, train classification head only |
| `full` | Unfreeze and update all weights |
| `lora` | Low-rank adaptation of attention layers (best accuracy/compute trade-off) |

A **bonus segmentation head** section shows how to exploit Pixio's patch tokens for dense prediction.

### Requirements
- `transformers >= 4.48.0` (Pixio support)
- `torch >= 2.0`
- No gated weights — Pixio checkpoints are **publicly available** on Hugging Face

## 1. Installation

In [ ]:
# Install / upgrade required libraries
!pip install -q --upgrade \
    'transformers>=4.48.0' \
    'torch>=2.0' \
    torchvision \
    huggingface_hub \
    peft \
    matplotlib \
    scikit-learn \
    tqdm

## 2. Imports & Global Config

In [ ]:
import os
import random
import math
import copy
import itertools
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from torchvision.datasets import ImageFolder

from transformers import AutoImageProcessor, AutoModel

print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device  : {device}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Global configuration — edit these to match your setup ─────────────────────
CFG = dict(
    # ---- Data ----------------------------------------------------------------
    # Root folder that contains one sub-folder per class:
    #   data_root/
    #       class_a/  img1.jpg  img2.png ...
    #       class_b/  img1.jpg  ...
    data_root       = "./data",         # <-- change to your dataset path
    val_split       = 0.15,             # fraction used for validation
    test_split      = 0.10,            # fraction used for test
    num_workers     = 4,

    # ---- Model ---------------------------------------------------------------
    # Available Pixio checkpoints on Hugging Face (all public, no gating):
    #   facebook/pixio-vitb16   (ViT-B, ~86 M params)  ← fast, good baseline
    #   facebook/pixio-vitl16   (ViT-L, ~300 M params)
    #   facebook/pixio-vith16   (ViT-H, 631 M params)  ← best student
    #   facebook/pixio-vit1b16  (ViT-1B, ~1 B params)
    #   facebook/pixio-vit5b16  (ViT-5B, 5 B params)   ← teacher, very large
    model_name      = "facebook/pixio-vitb16",
    num_classes     = None,            # auto-detected from folder structure

    # ── Pixio-specific: number of CLS tokens ──────────────────────────────────
    # Pixio uses 8 [CLS] tokens (prepended before patch tokens in the sequence).
    # All 8 are concatenated to form the global image representation.
    num_cls_tokens  = 8,

    # ---- Fine-tuning strategy: 'linear_probe' | 'full' | 'lora' -------------
    strategy        = "linear_probe",

    # ---- LoRA settings (used only when strategy == 'lora') ------------------
    lora_rank       = 8,
    lora_alpha      = 16,
    lora_dropout    = 0.05,

    # ---- Training ------------------------------------------------------------
    # Image size must be a multiple of the patch size (16).
    # Pixio was trained at 224×224; larger sizes work for dense tasks.
    image_size      = 224,
    batch_size      = 32,
    epochs          = 20,
    lr              = 1e-3,            # head learning rate
    backbone_lr     = 1e-5,           # backbone lr (used for 'full' strategy)
    weight_decay    = 1e-4,
    patience        = 5,              # early-stopping patience
    output_dir      = "./pixio_finetuned",
)

os.makedirs(CFG['output_dir'], exist_ok=True)
print("Config loaded:", CFG)

## 3. Dataset Preparation

Expects an **ImageFolder**-style directory layout:
```
data/
├── class_a/
│   ├── img001.jpg
│   └── ...
└── class_b/
    ├── img001.jpg
    └── ...
```

Pixio was pretrained with **ImageNet normalisation statistics**, so we reuse them here.

In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
IMG_MEAN = [0.485, 0.456, 0.406]   # ImageNet statistics
IMG_STD  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.RandomResizedCrop(CFG['image_size'], scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

val_transforms = T.Compose([
    T.Resize(int(CFG['image_size'] * 1.14)),
    T.CenterCrop(CFG['image_size']),
    T.ToTensor(),
    T.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

In [ ]:
# ── Build dataset splits ──────────────────────────────────────────────────────
full_dataset = ImageFolder(root=CFG['data_root'], transform=train_transforms)

CFG['num_classes'] = len(full_dataset.classes)
class_names = full_dataset.classes
print(f"Found {len(full_dataset)} images across {CFG['num_classes']} classes: {class_names}")

n_total = len(full_dataset)
n_val   = int(n_total * CFG['val_split'])
n_test  = int(n_total * CFG['test_split'])
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply val transforms to val / test splits
val_ds.dataset  = copy.copy(full_dataset)
val_ds.dataset.transform  = val_transforms
test_ds.dataset = copy.copy(full_dataset)
test_ds.dataset.transform = val_transforms

print(f"Train: {n_train} | Val: {n_val} | Test: {n_test}")

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

In [ ]:
# ── Visualise a few training samples ─────────────────────────────────────────
def denormalize(tensor, mean=IMG_MEAN, std=IMG_STD):
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(imgs): break
    ax.imshow(denormalize(imgs[i]).permute(1, 2, 0).numpy())
    ax.set_title(class_names[labels[i].item()], fontsize=9)
    ax.axis('off')
plt.suptitle("Sample training images", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Load Pixio Backbone

Pixio is loaded via HuggingFace `AutoModel`. Its output token sequence is:

```
[ CLS₁ | CLS₂ | … | CLS₈ | patch₁ | patch₂ | … | patchN ]
```

For a 224×224 image with patch size 16 → N = (224/16)² = 196 patch tokens.  
Total sequence length = **8 + 196 = 204** tokens.

In [ ]:
# Load the image processor
processor = AutoImageProcessor.from_pretrained(CFG['model_name'])

# Load the Pixio backbone
backbone = AutoModel.from_pretrained(CFG['model_name'])

embed_dim     = backbone.config.hidden_size
num_cls       = CFG['num_cls_tokens']
patch_size    = backbone.config.patch_size
num_patches   = (CFG['image_size'] // patch_size) ** 2

print(f"Model         : {CFG['model_name']}")
print(f"Embed dim     : {embed_dim}")
print(f"Patch size    : {patch_size}")
print(f"# CLS tokens  : {num_cls}")
print(f"# Patch tokens: {num_patches}  (for {CFG['image_size']}×{CFG['image_size']})")
print(f"Total tokens  : {num_cls + num_patches}")
print(f"Params        : {sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M")

In [ ]:
# ── Sanity-check: confirm token layout ───────────────────────────────────────
dummy = torch.zeros(1, 3, CFG['image_size'], CFG['image_size'])
with torch.no_grad():
    out = backbone(pixel_values=dummy)

seq_len = out.last_hidden_state.shape[1]
print(f"Sequence length (last_hidden_state): {seq_len}")
assert seq_len == num_cls + num_patches, (
    f"Expected {num_cls + num_patches} tokens, got {seq_len}. "
    "Adjust num_cls_tokens in CFG."
)
print("Token layout confirmed: 8 CLS + patch tokens.")

## 5. Build the Classification Model

The key Pixio advantage: **all 8 [CLS] tokens** carry complementary global information (scene, style, pose, …).  
We concatenate them with mean-pooled patch tokens to form a rich feature vector:

```
feature = concat(CLS₁..CLS₈, mean(patches))   →   shape: (B, 9 × embed_dim)
```

In [ ]:
class PixioClassifier(nn.Module):
    """Pixio backbone + lightweight classification head.

    Feature vector = concat of all 8 [CLS] tokens + mean-pooled patch tokens.
    Total input dim to head = (num_cls + 1) * embed_dim.
    """

    def __init__(
        self,
        backbone: nn.Module,
        num_classes: int,
        num_cls_tokens: int = 8,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.backbone       = backbone
        self.num_cls_tokens = num_cls_tokens
        embed_dim           = backbone.config.hidden_size
        feat_dim            = embed_dim * (num_cls_tokens + 1)  # 8 CLS + 1 mean-patch

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(pixel_values=pixel_values)
        # last_hidden_state: (B, seq_len, embed_dim)
        hs = outputs.last_hidden_state

        # First num_cls_tokens positions are [CLS] tokens
        cls_tokens   = hs[:, :self.num_cls_tokens, :]       # (B, 8, D)
        patch_tokens = hs[:, self.num_cls_tokens:, :]       # (B, N, D)

        # Flatten all CLS tokens and concatenate with mean-pooled patches
        cls_flat     = cls_tokens.reshape(cls_tokens.size(0), -1)  # (B, 8*D)
        mean_patch   = patch_tokens.mean(dim=1)                     # (B, D)
        features     = torch.cat([cls_flat, mean_patch], dim=1)     # (B, 9*D)

        return self.head(features)

In [ ]:
def build_model(strategy: str) -> nn.Module:
    """Return a PixioClassifier configured for the requested fine-tuning strategy."""

    if strategy == 'linear_probe':
        # Freeze the entire backbone — only the head is trainable
        for p in backbone.parameters():
            p.requires_grad = False
        model = PixioClassifier(backbone, CFG['num_classes'], CFG['num_cls_tokens'])

    elif strategy == 'full':
        # All parameters trainable; backbone uses a lower LR (set in optimizer)
        for p in backbone.parameters():
            p.requires_grad = True
        model = PixioClassifier(backbone, CFG['num_classes'], CFG['num_cls_tokens'])

    elif strategy == 'lora':
        from peft import get_peft_model, LoraConfig, TaskType

        # Freeze backbone first
        for p in backbone.parameters():
            p.requires_grad = False

        lora_cfg = LoraConfig(
            task_type      = TaskType.FEATURE_EXTRACTION,
            r              = CFG['lora_rank'],
            lora_alpha     = CFG['lora_alpha'],
            lora_dropout   = CFG['lora_dropout'],
            target_modules = ["query", "key", "value", "dense"],
            bias           = "none",
        )
        lora_backbone = get_peft_model(backbone, lora_cfg)
        lora_backbone.print_trainable_parameters()
        model = PixioClassifier(lora_backbone, CFG['num_classes'], CFG['num_cls_tokens'])

    else:
        raise ValueError(f"Unknown strategy '{strategy}'. Choose: linear_probe | full | lora")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"[{strategy}] Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params")
    return model.to(device)


model = build_model(CFG['strategy'])
print(model)

## 6. Optimizer, Scheduler & Loss

In [ ]:
def build_optimizer(model: nn.Module, strategy: str) -> optim.Optimizer:
    if strategy == 'full':
        backbone_params = [p for n, p in model.named_parameters()
                           if 'backbone' in n and p.requires_grad]
        head_params     = [p for n, p in model.named_parameters()
                           if 'head' in n and p.requires_grad]
        param_groups = [
            {'params': backbone_params, 'lr': CFG['backbone_lr']},
            {'params': head_params,     'lr': CFG['lr']},
        ]
    else:
        param_groups = [p for p in model.parameters() if p.requires_grad]
    return optim.AdamW(param_groups, lr=CFG['lr'], weight_decay=CFG['weight_decay'])


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = build_optimizer(model, CFG['strategy'])

# Cosine annealing with linear warm-up
warmup_epochs = max(1, CFG['epochs'] // 10)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs),
        optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                              T_max=CFG['epochs'] - warmup_epochs,
                                              eta_min=1e-7),
    ],
    milestones=[warmup_epochs],
)

print(f"Optimizer : AdamW | LR {CFG['lr']} | WD {CFG['weight_decay']}")
print(f"Scheduler : LinearWarmup ({warmup_epochs} ep) → CosineAnnealing")

## 7. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    """One epoch of training or validation. Returns (loss, accuracy)."""
    is_train = (phase == 'train')
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in tqdm(loader, desc=phase, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total

In [ ]:
history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
no_improve   = 0
best_ckpt    = os.path.join(CFG['output_dir'], 'best_model.pt')

print(f"\nTraining Pixio [{CFG['strategy']}] for {CFG['epochs']} epochs ...\n")

for epoch in range(1, CFG['epochs'] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, 'train')
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, None,      'val')
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    lr_now = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:>3}/{CFG['epochs']} "
          f"| LR {lr_now:.2e} "
          f"| Train loss {train_loss:.4f}  acc {train_acc:.4f} "
          f"| Val   loss {val_loss:.4f}  acc {val_acc:.4f}", end="")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve   = 0
        torch.save(model.state_dict(), best_ckpt)
        print("  ← best")
    else:
        no_improve += 1
        print(f"  (no improvement {no_improve}/{CFG['patience']})")

    if no_improve >= CFG['patience']:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest val accuracy : {best_val_acc:.4f}")
print(f"Checkpoint saved  : {best_ckpt}")

## 8. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_ran, history['train_acc'], label='Train')
axes[1].plot(epochs_ran, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle(
    f'Pixio Fine-tune ({CFG["strategy"]}) — {CFG["model_name"].split("/")[-1]}',
    fontsize=12
)
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'training_curves.png'), dpi=150)
plt.show()

## 9. Evaluation on Test Set

In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Evaluating test set'):
        imgs   = imgs.to(device)
        logits = model(imgs)
        preds  = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = (all_preds == all_labels).mean()
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Confusion matrix
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(max(6, CFG['num_classes']),
                                max(5, CFG['num_classes'] - 1)))
im = ax.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(CFG['num_classes']))
ax.set_yticks(range(CFG['num_classes']))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
thresh = cm_norm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.2f})",
            ha='center', va='center',
            color='white' if cm_norm[i, j] > thresh else 'black', fontsize=8)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
ax.set_title('Confusion Matrix (normalised)')
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'confusion_matrix.png'), dpi=150)
plt.show()

## 10. Inference on a Single Image

In [ ]:
def predict_image(image_path: str, model: nn.Module, transform, top_k: int = 3):
    """Run inference on a single image and return top-k predictions."""
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]

    topk_probs, topk_idxs = probs.topk(min(top_k, len(class_names)))
    results = [(class_names[i.item()], p.item()) for i, p in zip(topk_idxs, topk_probs)]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title('Input image')
    axes[0].axis('off')

    labels_ = [r[0] for r in results]
    scores_ = [r[1] for r in results]
    axes[1].barh(labels_[::-1], scores_[::-1])
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel('Probability')
    axes[1].set_title(f'Top-{top_k} Predictions')

    plt.tight_layout()
    plt.show()
    return results


# Example — replace with your image path
# results = predict_image('./my_image.jpg', model, val_transforms)
# print(results)

## 11. Save & Export the Fine-tuned Model

In [ ]:
# ── Save full checkpoint ──────────────────────────────────────────────────────
final_ckpt = os.path.join(CFG['output_dir'], 'pixio_classifier_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names'     : class_names,
    'num_classes'     : CFG['num_classes'],
    'strategy'        : CFG['strategy'],
    'model_name'      : CFG['model_name'],
    'num_cls_tokens'  : CFG['num_cls_tokens'],
    'embed_dim'       : embed_dim,
    'best_val_acc'    : best_val_acc,
}, final_ckpt)
print(f"Final checkpoint saved: {final_ckpt}")

# ── (Optional) Push backbone to Hugging Face Hub ──────────────────────────────
# backbone.push_to_hub("your-hf-username/pixio-finetuned-custom")
# processor.push_to_hub("your-hf-username/pixio-finetuned-custom")

In [ ]:
# ── Load checkpoint for downstream use ───────────────────────────────────────
def load_pixio_classifier(checkpoint_path: str):
    """Re-instantiate and load a saved PixioClassifier."""
    ckpt       = torch.load(checkpoint_path, map_location='cpu')
    _backbone  = AutoModel.from_pretrained(ckpt['model_name'])
    _model     = PixioClassifier(
        _backbone,
        num_classes    = ckpt['num_classes'],
        num_cls_tokens = ckpt['num_cls_tokens'],
    )
    _model.load_state_dict(ckpt['model_state_dict'])
    _model.eval()
    print(f"Loaded model | classes: {ckpt['class_names']} | val acc: {ckpt['best_val_acc']:.4f}")
    return _model, ckpt['class_names']


# loaded_model, loaded_classes = load_pixio_classifier(final_ckpt)
print("Load helper defined. Uncomment above line to test reloading.")

## 12. Bonus: Segmentation Head (Dense Prediction)

Pixio's pixel-level pretraining makes its **patch tokens** especially informative for dense tasks.  
This section shows a simple **linear segmentation head** that reshapes patch tokens into a 2-D feature map and up-samples to the input resolution — a strong baseline for semantic segmentation.

```
patch tokens (B, N, D)  →  reshape to (B, D, H/16, W/16)  →  bilinear upsample  →  conv 1×1  →  (B, C, H, W)
```

> **Note**: For production segmentation, replace the linear head with a DPT (Dense Prediction Transformer) head, as used in the Pixio paper.

In [ ]:
class PixioSegmentationHead(nn.Module):
    """Simple linear segmentation head on top of frozen Pixio patch tokens.

    Input image size must be divisible by patch_size (16).
    Outputs a per-pixel class map at the original resolution.
    """

    def __init__(
        self,
        backbone: nn.Module,
        num_seg_classes: int,
        image_size: int    = 224,
        num_cls_tokens: int = 8,
    ):
        super().__init__()
        self.backbone       = backbone
        self.num_cls_tokens = num_cls_tokens
        self.patch_size     = backbone.config.patch_size
        self.h_patches      = image_size // self.patch_size   # e.g. 14 for 224/16
        self.w_patches      = self.h_patches
        embed_dim           = backbone.config.hidden_size

        self.head = nn.Sequential(
            nn.Conv2d(embed_dim, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Conv2d(256, num_seg_classes, kernel_size=1),
        )

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        B, C, H, W = pixel_values.shape
        outputs = self.backbone(pixel_values=pixel_values)
        hs = outputs.last_hidden_state                       # (B, 8+N, D)

        # Drop CLS tokens, keep only patch tokens
        patch_tokens = hs[:, self.num_cls_tokens:, :]        # (B, N, D)

        # Reshape to spatial grid
        D = patch_tokens.size(-1)
        feat_map = (
            patch_tokens
            .permute(0, 2, 1)                               # (B, D, N)
            .reshape(B, D, self.h_patches, self.w_patches)  # (B, D, h, w)
        )

        # Upsample to original resolution and predict
        feat_map = nn.functional.interpolate(
            feat_map, size=(H, W), mode='bilinear', align_corners=False
        )
        return self.head(feat_map)                           # (B, num_seg_classes, H, W)

In [ ]:
# ── Quick forward-pass sanity check (frozen backbone) ────────────────────────
NUM_SEG_CLASSES = 21   # e.g. Pascal VOC — change to your number of semantic classes

# Freeze backbone for linear probing
seg_backbone = AutoModel.from_pretrained(CFG['model_name'])
for p in seg_backbone.parameters():
    p.requires_grad = False

seg_model = PixioSegmentationHead(
    seg_backbone,
    num_seg_classes = NUM_SEG_CLASSES,
    image_size      = CFG['image_size'],
    num_cls_tokens  = CFG['num_cls_tokens'],
).to(device)

dummy_img   = torch.zeros(2, 3, CFG['image_size'], CFG['image_size']).to(device)
dummy_logits = seg_model(dummy_img)
print(f"Seg output shape: {dummy_logits.shape}")
# Expected: (2, NUM_SEG_CLASSES, 224, 224)

trainable = sum(p.numel() for p in seg_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in seg_model.parameters())
print(f"Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M params (head only)")

# To train: swap dummy_img for real images and dummy_logits for cross-entropy
# seg_criterion = nn.CrossEntropyLoss(ignore_index=255)  # 255 is typically the ignore label
# seg_loss = seg_criterion(dummy_logits, seg_masks)  # seg_masks: (B, H, W) LongTensor
print("\nSegmentation head ready. Plug in your segmentation DataLoader to train.")

## Summary

### Fine-tuning Strategies (Classification)

| Strategy | Trainable params | Compute | Typical accuracy |
|---|---|---|---|
| `linear_probe` | ~1–4 K (head only) | Very low | Good baseline |
| `lora` | ~1–3 M (LoRA + head) | Low–medium | Near full fine-tune |
| `full` | All (~86–631 M) | High | Highest, risk of overfitting |

### Pixio vs DINOv3 for Fine-tuning

| | Pixio | DINOv3 |
|---|---|---|
| CLS tokens | **8** (richer global repr.) | 1 |
| Best for | Dense tasks + classification | Classification + retrieval |
| Gated weights | **No** (public) | Yes |
| Pretraining | Pixel reconstruction (MAE-style) | Distillation |
| Feature vector | concat(8×CLS, mean_patch) | concat(CLS, mean_patch) |

### Tips
- **Start** with `linear_probe` to get a baseline quickly.
- **Move to** `lora` for best accuracy/compute trade-off on small datasets (< 10 k images).
- **Use** `full` fine-tuning only with large datasets and sufficient GPU memory.
- **Dense tasks** (depth, segmentation): use only patch tokens reshaped to a 2-D feature map.
- Pixio's patch tokens are especially powerful due to pixel-level pretraining supervision.

### References
- [Pixio: Pixel Supervision for Visual Pre-training (arXiv:2512.15715)](https://arxiv.org/abs/2512.15715)
- [GitHub: facebookresearch/pixio](https://github.com/facebookresearch/pixio)
- [Hugging Face: facebook/pixio-vith16](https://huggingface.co/facebook/pixio-vith16)
- [Transformers docs — Pixio](https://huggingface.co/docs/transformers/main/model_doc/pixio)